In [ ]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from read_data import save_files

### Read Data

In [ ]:
def read_data():
    return [
        pd.read_csv('../data/01-starting_data/development_data/awards_players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/coaches.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/series_post.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

Section where we select relevant data and filter out invariant or irrelevant columns 

In [ ]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=['firstseason', 'lastseason', 'college', 'collegeOther', 'deathDate'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series' too ??
teams = teams.drop(columns=['lgID', 'franchID', 'divID', 'arena', 'name', 'seeded', 
                        'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
                        'firstRound', 'semis', 'finals'])
teams_post = teams_post.drop(columns=['lgID'])

In [ ]:
save_files("02-data_selection", 
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"], 
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

### Data Preparation

Section where we treat cases like non-existing values, outliers, etc.

In [ ]:
save_files("03-data_preparation",
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

### Feature Engineering

Section where we create and/or simplify existing metrics to aid the prediction model

In [ ]:
# def calculateUPER():
#     data['uPER'] = 

# def calculatePER():
#     data['PER'] = uPER * (lgPace/tmPace) * (15 / lguPER) 

In [ ]:
teams['win_loss_ratio'] = round(teams['won'] / (teams['won'] + teams['lost']),3)  
teams['confIDbin'] = teams['confID'].apply(lambda x: 1 if x == 'EA' else 0)

teams['playoff_qualification'] = teams['playoff'].apply(lambda x: 1.0 if x == 'Y' else 0.0)
teams.drop(columns=['playoff'], inplace=True)

In [ ]:
awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')
players_teams = players_teams.merge(awards_count, on=['playerID', 'year'], how='left')
players_teams['num_awards'] = players_teams['num_awards'].fillna(0)

In [ ]:
save_files("04-feature_engineering",
            ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
            [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

### Add Year 11 info

In [ ]:
def read_data11():
    return [
        pd.read_csv('../data/01-starting_data/challenge/coaches.csv'),
        pd.read_csv('../data/01-starting_data/challenge/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/challenge/teams.csv')
    ]

coaches11, players_teams11, teams11 = read_data11()

coaches11 = coaches11.drop(columns=['lgID'])
players_teams11 = players_teams11.drop(columns=['lgID'])
teams11 = teams11.drop(columns=['lgID', 'franchID', 'arena', 'name'])

teams = pd.concat([teams, teams11], ignore_index=True)
players_teams = pd.concat([players_teams, players_teams11], ignore_index=True)
coaches = pd.concat([coaches, coaches11], ignore_index=True)


In [ ]:
def shift_performance(data, naValueMode, unvariant, columns):
    """
    Shifts the performance metrics for next year
    Fills non-existing data with the mean of the year
    
    naValueMode- 0: fill with 0, 1: fill with 15% quantile, 2: fill with mean
    0 - values must be 0 due to context
    1 - rookie values (below average) (if there is no data, player/coach is rookie)
    2 - average values
    """
    #Shift data
    for column in columns:
        data = data.assign(**{column: data.groupby(unvariant)[column].shift(1)})
    #drop data from year 1
    data = data[data['year'] != 1]
    
    #Fill missing data with appropriate values
    for column in columns:

        if column == 'playoff_qualification':
            data[column] = data[column].fillna(0)
            continue

        year_data_mean = data.groupby('year')[column].mean()
        year_data_quantile = data.groupby('year')[column].quantile(0.15)
        for year in data['year'].unique():
            if naValueMode == 0:
                data.loc[data['year'] == year, column] = data.loc[data['year'] == year, column].fillna(0)
            elif naValueMode == 1:
                data.loc[data['year'] == year, column] = data.loc[data['year'] == year, column].fillna(round(year_data_quantile[year],2))
            elif naValueMode == 2:
                data.loc[data['year'] == year, column] = data.loc[data['year'] == year, column].fillna(round(year_data_mean[year],2))

    return data

awards_players['year'] = awards_players['year'] + 1
coaches = shift_performance(coaches, 1,['coachID'], [column for column in coaches.columns if column not in ["coachID","year","tmID", 'stint']])
players_teams = shift_performance(players_teams, 1,['playerID'], [column for column in players_teams.columns if column not in ["playerID","year","stint","tmID"]])
series_post['year'] = series_post['year'] + 1
teams_post['year'] = teams_post['year'] + 1
teams = shift_performance(teams, 2, ['tmID'], [column for column in teams.columns if column not in ['year', 'tmID', 'confID', 'confIDbin', 'playoff']])

#for debug
#save_files("05-data_shift",
#            ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
#            [awards_players, coaches, players, players_teams, series_post, teams, teams_post])


### Data Merging

Section responsible for merging all tables, in a format ready to feed the model

In [ ]:
#team metrics
data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum'
}).reset_index()
data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()
data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")

data.columns

In [ ]:
#write df to csv
save_files("05-processed_data", ["processed_data"], [data])

data.fillna(0, inplace=True)

print(f"final data columns {data.columns}")

### Model training

In [ ]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVR

#### Initialization 

In [ ]:
#result lists
accuracy_scores = []
error_scores = []

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['playoff_qualification', 'tmID', 'confID']]
target_column = 'playoff_qualification'

#Create model
#model = DecisionTreeClassifier(random_state=21) #acc: 0.55 error: 0.59 || acc: 0.59  error: 0.55
model = RandomForestClassifier(random_state=21) #acc: 0.60 error: 0.43 || acc: 0.64  error: 0.44
#model = LogisticRegression(random_state=21)     #acc: 0.64 error: 0.39 || acc: 0.57  error: 0.44
#model = SVR()                                   #acc: 0.62 error: 0.44 || acc: 0.60  error: 0.45

data.columns

#### Year training cycle

In [ ]:
for year in sorted(data['year'].unique()):  # Start from the second year (with )
    # Separate train and test data
    year_span = 3
    train_data = data[data['year'] <= year] if year < (year_span - 1) else data[(data['year'] <= year) & (data['year'] >= year - (year_span - 1))]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip (redundant?)
    if test_data.empty:
        continue

    y_test = test_data[target_column]

    results_df = pd.DataFrame()
    for confID in data['confID'].unique():
        conf_train_data = train_data[train_data['confID'] == confID]
        conf_test_data = test_data[test_data['confID'] == confID]
    
        
        confIDs = conf_test_data['confID'].values
        teamIDs = conf_test_data['tmID'].values
    
        conf_X_train = conf_train_data[feature_columns]
        conf_y_train = conf_train_data[target_column]
    
        conf_X_test = conf_test_data[feature_columns]
        conf_y_test = conf_test_data[target_column]
        
        # Standardize the data
        scaler = StandardScaler()
        conf_X_train = scaler.fit_transform(conf_X_train)
        conf_X_test = scaler.transform(conf_X_test)
    
        # Train the model
        model.fit(conf_X_train, conf_y_train)
    
        # Make predictions on the test set
        y_pred_proba = model.predict_proba(conf_X_test)[:,1]
        #y_pred_proba_norm = (y_pred_proba - min(y_pred_proba)) / (max(y_pred_proba) - min(y_pred_proba)) # Normalize to 0-1
        y_pred_proba_scale = y_pred_proba * 4 / sum(y_pred_proba)  # Normalize to sum to 4
    
    
        conf_results_df = pd.DataFrame({
            'tmID': teamIDs,
            'confID': confIDs,
            'Playoff': np.round(y_pred_proba_scale, 2)
        })
    
        conf_y_pred = np.zeros_like(y_pred_proba_scale)
        top_4_indices = conf_results_df.nlargest(4, 'Playoff').index
        conf_y_pred[top_4_indices] = 1
    
        conf_results_df['Label'] = conf_y_pred
    
        results_df = pd.concat([results_df, conf_results_df])
        results_df.sort_values(by='tmID', ascending=True, inplace=True)
    
    
    y_pred = results_df['Label']
    y_pred_proba_norm = results_df['Playoff']

    if (year == 10):
        #results_df = results_df.drop(columns=['Label', 'confID'])
        results_df.to_csv('../data/06-results/results.csv', index=False)
    # Calculate accuracy and error

    if year < 10:
        accuracy_scores.append(round(accuracy_score(y_test, y_pred),2))

        error_array = np.abs(y_pred_proba_norm - y_test.values)
        error_score = round(sum(error_array),2)
        error_scores.append(error_score)

        # Output results for each year
        print(f"Year {year} -> {year + 1}:")
        print(f"Results: \n predict: \n {results_df['Playoff'].values}\n label: \t {y_pred.values}\n expected: {y_test.values}\n error: \t {error_array.values}")
        print(f"  Accuracy: {accuracy_scores[-1]}")
        print(f"  Error: \t {round(sum(error_array), 2)} / {len(error_array)}")
        print("\n")

### End Results

In [ ]:
print(f"Accuracy  {accuracy_scores}")
print(f"Error \t {[float(e) for e in error_scores]}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores) / len(accuracy_scores):.2f}")
print(f"  Average Error: \t {round(sum(error_scores) / len(error_scores),2)}")